# Dataset structure statistics

This notebook scans the dataset folder and reports case-level, side-level, visit-level, and image-level statistics.

In [1]:
from pathlib import Path
from collections import Counter
import json
import pandas as pd

try:
    from IPython.display import display
except ImportError:
    display = print

ROOT = Path(r"D:/github_codes/3合并12批数据/raw")
IMAGE_EXTS = {".png", ".jpg", ".jpeg", ".bmp", ".tif", ".tiff", ".dcm"}

assert ROOT.exists(), f"Dataset root does not exist: {ROOT}"

In [ ]:
case_rows = []
side_rows = []
visit_rows = []
image_rows = []

case_dirs = sorted([p for p in ROOT.iterdir() if p.is_dir() and p.name.startswith("case_")])

for case_dir in case_dirs:
    case_meta_path = case_dir / "case_meta.json"
    case_meta = json.loads(case_meta_path.read_text(encoding="utf-8")) if case_meta_path.exists() else {}
    case_id = case_meta.get("case_id", case_dir.name)
    left_label = int(case_meta.get("left_label", 0))
    right_label = int(case_meta.get("right_label", 0))
    case_label = 1 if (left_label == 1 or right_label == 1) else 0
    case_visit_count = 0
    case_image_count = 0

    for side, label in [("left", left_label), ("right", right_label)]:
        side_rows.append({
            "case_id": case_id,
            "side": side,
            "label": label,
            "label_name": "positive" if label == 1 else "negative",
        })

    for phase_dir in sorted([p for p in case_dir.iterdir() if p.is_dir()]):
        if phase_dir.name not in {"1prenatal", "2postnatal"}:
            continue

        for visit_dir in sorted([p for p in phase_dir.iterdir() if p.is_dir()]):
            visit_meta_path = visit_dir / "meta.json"
            visit_meta = json.loads(visit_meta_path.read_text(encoding="utf-8")) if visit_meta_path.exists() else {}
            visit_id = visit_meta.get("visit_id", visit_dir.name)
            phase = visit_meta.get("phase", phase_dir.name)
            exam_date = visit_meta.get("exam_date")
            left_image_count = 0
            right_image_count = 0

            for side in ["left", "right"]:
                image_dir = visit_dir / side / "images"
                images = sorted([p for p in image_dir.iterdir() if p.is_file() and p.suffix.lower() in IMAGE_EXTS]) if image_dir.exists() else []
                if side == "left":
                    left_image_count = len(images)
                else:
                    right_image_count = len(images)

                for image_path in images:
                    image_rows.append({
                        "case_id": case_id,
                        "visit_id": visit_id,
                        "phase": phase,
                        "exam_date": exam_date,
                        "side": side,
                        "image_name": image_path.name,
                        "image_path": str(image_path),
                    })

            total_image_count = left_image_count + right_image_count
            case_visit_count += 1
            case_image_count += total_image_count
            visit_rows.append({
                "case_id": case_id,
                "visit_id": visit_id,
                "phase": phase,
                "exam_date": exam_date,
                "left_image_count": left_image_count,
                "right_image_count": right_image_count,
                "total_image_count": total_image_count,
            })

    case_rows.append({
        "case_id": case_id,
        "left_label": left_label,
        "right_label": right_label,
        "case_label": case_label,
        "case_label_name": "positive" if case_label == 1 else "negative",
        "visit_count": case_visit_count,
        "image_count": case_image_count,
    })

case_df = pd.DataFrame(case_rows)
side_df = pd.DataFrame(side_rows)
visit_df = pd.DataFrame(visit_rows)
image_df = pd.DataFrame(image_rows)

visit_df["exam_date"] = pd.to_datetime(visit_df["exam_date"], errors="coerce")
image_df["exam_date"] = pd.to_datetime(image_df["exam_date"], errors="coerce")

case_df.head(), visit_df.head(), image_df.head()

In [ ]:
# Overall answers
case_label_counts = case_df["case_label_name"].value_counts().reindex(["positive", "negative"], fill_value=0)
side_label_counts = side_df["label_name"].value_counts().reindex(["positive", "negative"], fill_value=0)

years = sorted(visit_df["exam_date"].dropna().dt.year.unique().tolist())
year_span = f"{min(years)}-{max(years)}" if years else "unknown"

summary = pd.DataFrame([
    {"item": "total_cases", "value": len(case_df), "note": "case_xxxx directories"},
    {"item": "case_positive", "value": int(case_label_counts["positive"]), "note": "case-level: left_label=1 or right_label=1"},
    {"item": "case_negative", "value": int(case_label_counts["negative"]), "note": "case-level: both left_label=0 and right_label=0"},
    {"item": "side_positive", "value": int(side_label_counts["positive"]), "note": "side-level: left/right counted separately"},
    {"item": "side_negative", "value": int(side_label_counts["negative"]), "note": "side-level: left/right counted separately"},
    {"item": "total_visits", "value": len(visit_df), "note": "visit/exam directories"},
    {"item": "total_images", "value": len(image_df), "note": "image files under left/right/images"},
    {"item": "year_span", "value": year_span, "note": f"years found: {years}"},
    {"item": "center_count", "value": 1, "note": "single-center, based on your provided description"},
])

summary

In [ ]:
# How many images and visits does each patient/case have?
case_df.sort_values("case_id")

In [ ]:
# How many images does each visit/exam have?
visit_df.sort_values(["case_id", "exam_date", "visit_id"])

In [ ]:
# Helpful distributions for reporting ranges/median/IQR if needed
distributions = {
    "images_per_case": case_df["image_count"].describe(),
    "visits_per_case": case_df["visit_count"].describe(),
    "images_per_visit": visit_df["total_image_count"].describe(),
}

for name, stats in distributions.items():
    print(f"\n{name}")
    print(stats)

zero_image_visits = visit_df[visit_df["total_image_count"] == 0]
print(f"\nzero_image_visits: {len(zero_image_visits)}")
display(zero_image_visits.sort_values(["case_id", "exam_date", "visit_id"]))

In [ ]:
# Optional: save detail tables next to the notebook
OUT_DIR = Path.cwd() / "dataset_statistics_outputs"
OUT_DIR.mkdir(exist_ok=True)

summary.to_csv(OUT_DIR / "summary.csv", index=False, encoding="utf-8-sig")
case_df.to_csv(OUT_DIR / "case_level_statistics.csv", index=False, encoding="utf-8-sig")
visit_df.to_csv(OUT_DIR / "visit_level_statistics.csv", index=False, encoding="utf-8-sig")
side_df.to_csv(OUT_DIR / "side_level_labels.csv", index=False, encoding="utf-8-sig")

OUT_DIR